# Monitoreo agroclimático de Honduras — FAO GIEWS / ASIS

**Análisis municipal de estrés hídrico agrícola y condición de la vegetación, 2005 en adelante.**

Este cuaderno reconstruye, desde las fuentes primarias de FAO, un panel municipio × dekada de indicadores agroclimáticos para Honduras, lo valida contra la serie oficial publicada por GIEWS y lo usa para leer dos eventos históricos y el estado más reciente disponible.

| | |
|---|---|
| **Fuentes** | ImageServer ASIS (rásteres ~1 km) y CSV oficiales del portal GIEWS |
| **Unidad de análisis** | Municipio (GAUL 2015 nivel 2) y departamento (nivel 1) |
| **Resolución temporal** | Dekada (tercios de mes) |
| **Periodo histórico** | Desde 2005, controlado por la constante `ANIO_INICIO` |
| **Salidas** | Figuras HTML y tablas parquet/CSV en `./salidas`; caché reutilizable en `./asis_cache` |

## Cómo se recorre

El cuaderno está pensado para ejecutarse **de principio a fin y en orden**: cada celda usa objetos definidos en las anteriores. Las secciones 1 a 5 construyen la infraestructura y no producen conclusiones; las secciones 6 a 12 son el análisis; la 13 verifica que las cifras citadas en los textos de las figuras se reproduzcan.

| Parte | Secciones | Qué contiene |
|---|---|---|
| I. Infraestructura | 1–3 | Entorno, cliente de datos, geometrías y estadística zonal |
| II. Control de calidad | 4–5 | Validación contra el dato oficial y capa de visualización |
| III. Caso 1 — déficit | 6–7 | Sequía agrícola de la temporada primera 2019 |
| IV. Caso 2 — exceso | 8–9 | Huracanes Eta e Iota, noviembre 2020 |
| V. Estado actual | 10–12 | Últimos 18 meses publicados y cierre |
| VI. Verificación | 13 | Chequeos numéricos y limitaciones |

## Antes de ejecutar

La primera ejecución descarga varios miles de rásteres recortados y puede tardar. Todo lo descargado se guarda en `./asis_cache`, de modo que las corridas siguientes son rápidas y no vuelven a golpear los servicios de FAO. Si `www.fao.org` responde 403, el bloqueo viene de Cloudflare: el cuaderno indica en pantalla cómo colocar el CSV descargado a mano en la caché.

Cada figura ocupa su propia celda, de modo que una salida es siempre un solo gráfico y no hace falta desplazarse para verla completa. Las celdas que solo calculan no producen nada visible y son rápidas de saltar.

Las figuras se generan con Plotly y son interactivas; en un visor estático (GitHub, por ejemplo) no se verán. Este cuaderno se distribuye sin salidas guardadas, así que hay que ejecutarlo para ver los resultados.


## Cómo leer los indicadores

Antes de mirar cualquier figura conviene tener claro qué mide cada índice, porque los tres se confunden con facilidad y ninguno significa lo que su nombre sugiere en lenguaje corriente.

**Dekada.** ASIS trabaja en tercios de mes: D1 son los días 1 a 10, D2 del 11 al 20 y D3 del 21 al fin de mes. El año tiene 36 dekadas. La notación del cuaderno es `2019-09-D2`.

**ASI (Agricultural Stress Index).** Porcentaje del área de cultivo de la unidad que estuvo bajo estrés hídrico durante la temporada. Es **acumulativo dentro de la temporada** y se reinicia al comenzar la siguiente: cuando la temporada termina, el valor se congela hasta el reinicio. Mide déficit; por construcción no puede detectar un exceso de agua. Va de 0 a 100.

**VCI (Vegetation Condition Index).** Posición del vigor de la vegetación observada frente a su propio historial reciente, de 0 a 1. Cubre todo el territorio y todo el año, también fuera del área de cultivo. El umbral de alerta de FAO es 0,35.

**VHI (Vegetation Health Index).** Combina condición de la vegetación y temperatura. Aparece en el inventario de fuentes pero no se usa en el análisis.

**Temporadas agrícolas.** La *primera* se siembra entre mayo y junio y se cosecha entre agosto y septiembre (GS1); la *postrera* se siembra en septiembre y se cosecha entre diciembre y enero (GS2). Fuera de su ventana, el ASI de esa temporada no existe.

**Banderas 251 a 255.** Los rásteres no traen solo valores del índice: fuera del rango válido codifican situaciones (fuera de temporada, sin dato, sin estacionalidad, sin cultivo o pasto, nodata). Contarlas como si fueran ceros es el error más común al hacer estadística zonal, y aquí se evita enmascarándolas en el servidor y verificándolo en la sección 4.

Un municipio en blanco en los mapas no es un municipio sin estrés: es un municipio **sin dato** en esa dekada, casi siempre porque está fuera de su ventana de cultivo.


## Ventana temporal: 2005 en adelante

Todo el análisis histórico —climatología, líneas base, percentiles y series de lluvia— arranca en 2005 y está gobernado por una sola constante, `ANIO_INICIO`, definida en la sección 1. No hay años escritos a mano en ninguna figura: los títulos y subtítulos se construyen con f-strings sobre esa constante y sobre los datos efectivamente cargados, de modo que mover el periodo es cambiar un número y volver a ejecutar.

El corte tiene una razón sustantiva además de práctica: la serie de lluvia de GIEWS presenta un quiebre de homogeneidad alrededor de 2005, así que encadenarla con los años previos introduce un salto que no es climático. El costo es que la línea base de percentiles descansa ahora en catorce años (2005–2018) en lugar de treinta y cinco. Es suficiente para p10, mediana y p90, y escaso para hablar de colas extremas; conviene tenerlo presente al interpretar la sección 6.3 y la comparación histórica de la sección 12.

Lo que **no** se recorta: el inventario de fuentes de la sección 4 informa la cobertura completa de cada CSV, porque es un chequeo de salud del dato original; y el promedio histórico de lluvia (LTA) es el que publica FAO y no se recalcula, ya que es la referencia oficial contra la que GIEWS compara.


---

# Parte I. Infraestructura del análisis

Las tres secciones siguientes no producen ninguna conclusión: construyen las herramientas. Pueden ejecutarse y olvidarse, pero conviene leer al menos los parámetros de la sección 1, porque de ahí salen las decisiones que después no se vuelven a discutir.

## 1. Entorno, configuración y calendario dekadal

Instala lo que falte, fija los endpoints de FAO, define la malla nativa de ASIS y monta el calendario dekadal que se usa en todo el cuaderno.

Tres decisiones se toman aquí y valen para todo lo demás. La primera es el **periodo de análisis** (`ANIO_INICIO = 2005`). La segunda es el **rango válido de cada indicador**: fuera de él los rásteres traen banderas, no valores, y por eso se declaran explícitamente en `VALID_RANGE` y `FLAGS`. La tercera es la **caché en disco** (`./asis_cache`): todo lo que se descarga queda guardado, así el cuaderno es re-ejecutable sin volver a golpear los servicios de FAO.

También se definen aquí las clases y la paleta oficiales de FAO, que se reutilizan en todas las figuras para que los colores signifiquen siempre lo mismo.

*Produce:* constantes globales, sesión HTTP con reintentos, funciones de calendario dekadal. Sin salidas visuales.


In [ ]:
# CELDA 1 · Infraestructura: se importa del paquete
#
# El cálculo vive una sola vez, en `asis/`. Este cuaderno lo importa, de modo
# que una corrección en la estadística zonal se hace en un solo lugar y no en
# dos.
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "asis").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from asis import client, panel
from asis.config import (ASI_THRESHOLDS, CACHE, CLASSES, FLAGS, GRID_STEP,
                         HND_BBOX, NODATA, PALETTE, PIX_DEG, PIX_KM2, SEASONS,
                         SEASON_WINDOW, SERIES, SOURCE_NOTE, SOURCE_URL,
                         START_YEAR,
                         VALID_RANGE, WKID, in_season)
from asis.calendar import (MONTH_ES, dekad_between, dekad_code, dekad_date,
                           dekad_label, dekad_of_year, dekad_range,
                           dekad_window)
from asis.client import (SNAP, catalog_parsed, clip_period, export_tif,
                         export_tifs, last_dekad, load_csv, raster_name)
from asis.zonal import (SHAPE, TRANSFORM, export_geometry, grid,
                        municipal_series, read_tif, zonal_stats)
from asis.aggregate import (classify, climatology, department_weights,
                            national_from_csv, severity_area, to_country,
                            to_department, worst_case)
from asis.viz import (SCALE_ASI, SCALE_VCI, class_map, climatology_fig,
                      climatology_matrix, continuous_map, dashboard_fig,
                      heatmap_panel, rainfall_fig, ranking_fig, raster_detail,
                      series_fig, severity_area_fig, style_fig)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# La malla de zonas se arma una vez y la reutilizan todos los dekads: es lo que
# hace viable recorrer veinte años de rásteres.
G = grid()
MUNI, DEPT, NZ, ZONES = G.muni, G.dept, G.nz, G.zones

# Geometría simplificada para las coropletas (~400 m: mantiene la forma y pesa
# unas diez veces menos que la original).
GEOJSON, MUNI_REF = export_geometry()

print("infraestructura lista ·", pd.Timestamp.today().date(),
      "· periodo desde", START_YEAR, "·", NZ, "municipios ·",
      len(DEPT), "departamentos")
print("último dekad publicado · ASI_D GS1 LC-C:",
      last_dekad("ASI_D", season="GS1", landcover="C"),
      "· VCI_D:", last_dekad("VCI_D"))


---

# Parte II. Control de calidad

## 4. Validación contra el dato oficial de FAO

Ninguna cifra de este cuaderno debería publicarse sin pasar por aquí. La sección responde tres preguntas en orden.

Primero, **qué trae cada CSV oficial**: filas totales, filas dentro del periodo de análisis, cobertura temporal y nulos. Segundo, **cuánto del territorio es bandera y no dato**: se descarga el mismo ráster con y sin máscara y se verifica que el conteo de píxeles válidos coincida, lo que confirma que el enmascarado del servidor hace lo que se espera. Tercero, y es lo importante, **si el agregado municipal propio reproduce la serie departamental oficial**: se comparan dos temporadas separadas en el tiempo (2019 y 2023) y se reportan correlación, error absoluto medio y sesgo.

Si esa comparación se deteriorara en una ejecución futura, es señal de que algo cambió en la fuente o en la malla, y el resto del cuaderno deja de ser confiable.

*Produce:* inventario de fuentes, tabla de banderas, métricas de validación y un diagrama de dispersión contra la identidad.


### 4.1 Inventario de las series CSV

Qué trae cada fuente oficial: filas totales, filas dentro del periodo de análisis, cobertura temporal, provincias y nulos. La cobertura se informa completa a propósito, porque aquí se está auditando la fuente y no el recorte.

### 4.2 Cuánto del territorio es bandera y no dato

Se descarga el mismo ráster con máscara y sin ella. La tabla reparte los píxeles del país entre dato válido del índice y cada una de las banderas; el conteo de píxeles con dato debe coincidir en las dos versiones, y esa coincidencia es la que confirma que el enmascarado del servidor hace lo que se espera.

In [ ]:
# --- 4.2 Banderas vs dato -----------------------------------------------------
PEAK19, SUF_GS1, SUF_GS2 = "2019-09-D2", ".GS1.LC-C", ".GS2.LC-C"
_nom = raster_name("ASI_D", PEAK19, SUF_GS1)
crudo = read_tif(export_tif("ASI_D", _nom, mask=False))
enmas = read_tif(export_tif("ASI_D", _nom, mask=True))

dentro = ZONES > 0
v = crudo[dentro]
v = v[np.isfinite(v)]
etiqueta = np.where(v <= 100, "dato valido (0-100)",
                    pd.Series(np.round(v)).map(FLAGS).fillna("otro").values)
banderas = (pd.Series(etiqueta).value_counts(normalize=True).mul(100).round(2)
              .rename("% de pixeles del pais").to_frame())
banderas["km2"] = (pd.Series(etiqueta).value_counts() * PIX_KM2).round(0)
display(banderas)
print(f"pixeles con dato · crudo filtrado en [0,100]: {(v <= 100).sum():,} · "
      f"raster enmascarado por el servidor: {np.isfinite(enmas[dentro]).sum():,} "
      "(deben coincidir)")


### 4.3 ¿El agregado municipal reproduce la serie oficial?

La prueba de fondo. Se agrega el panel municipal propio a departamento y se compara contra lo que publica GIEWS, sobre dos temporadas separadas en el tiempo (2019 y 2023) para que la muestra no dependa de un solo año. Se reportan correlación, error absoluto medio y sesgo.

In [ ]:
# --- 4.3 Validacion municipio -> departamento contra el CSV oficial ----------
# Muestra independiente: dos temporadas de primera separadas en el tiempo.
DK_VALID = dekad_range(2019, 6, 1, 2019, 10, 1) + dekad_range(2023, 6, 1, 2023, 9, 3)
val_mun = municipal_series(SERIES["asi_gs1"], DK_VALID)
val_dep = to_department(val_mun)

ofi = load_csv("asi_dekad_s1")
ofi = ofi[ofi["Land_Type"].astype(str).str.contains("Crop", case=False, na=False)]
ofi = ofi[["ADM1_CODE", "Province", "dekad_id", "Data"]].dropna(subset=["Data"])
ofi["adm1_code"] = ofi["ADM1_CODE"].astype("int64").astype(str)

cmp_ = val_dep.merge(ofi[["adm1_code", "Province", "dekad_id", "Data"]],
                     on=["adm1_code", "dekad_id"], how="inner")
cmp_ = cmp_.rename(columns={"mean": "asi_raster", "Data": "asi_oficial"})
cmp_["dif"] = cmp_["asi_raster"] - cmp_["asi_oficial"]

r = np.corrcoef(cmp_["asi_raster"], cmp_["asi_oficial"])[0, 1]
mae = cmp_["dif"].abs().mean()
sesgo = cmp_["dif"].mean()
print(f"\nvalidacion · n={len(cmp_)} pares departamento-dekada · "
      f"r={r:.3f} (R2={r**2:.3f}) · MAE={mae:.2f} pp · sesgo={sesgo:+.2f} pp")
display(cmp_.groupby("dekad_id")[["asi_raster", "asi_oficial", "dif"]]
            .mean().round(2).head(20))


### 4.4 La misma comparación, punto por punto

Cada punto es un par departamento-dekada. La diagonal punteada es la identidad: mientras más pegada esté la nube, mejor reproduce el agregado propio al dato oficial. Las desviaciones sistemáticas de un departamento concreto apuntan a un problema de geometría, no de método.

In [ ]:
# --- 4.4 Validacion: dispersion contra la identidad --------------------------
lim = max(cmp_[["asi_raster", "asi_oficial"]].max()) * 1.05
fig_val = px.scatter(cmp_, x="asi_oficial", y="asi_raster", color="adm1_name",
                     hover_data={"dekad_id": True, "dif": ":.1f"},
                     labels={"asi_oficial": "ASI oficial GIEWS (% del area de cultivo)",
                             "asi_raster": "ASI reconstruido desde el raster",
                             "adm1_name": "Departamento"},
                     height=560, opacity=0.8)
fig_val.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim,
                  line=dict(color="#444", dash="dash"))
fig_val.update_traces(marker=dict(size=8, line=dict(width=0.4, color="white")))
fig_val.update_layout(
    template="plotly_white",
    title=dict(text="<b>Validacion: agregado municipal propio vs serie oficial FAO</b>"
                    f"<br><span style='font-size:12px;color:#555'>ASI temporada primera · "
                    f"{len(cmp_)} pares departamento-dekada · r={r:.3f} · MAE={mae:.2f} pp"
                    "</span>", x=0.01, xanchor="left"),
    margin=dict(l=60, r=20, t=90, b=60))
fig_val.show()


### 4.5 Cobertura del panel municipal

Cuántos municipios y cuántos kilómetros cuadrados tienen dato en cada dekada del periodo de validación. Sirve para descartar que un buen ajuste venga de comparar muy pocas unidades.

In [ ]:
# --- 4.5 Cobertura del panel municipal ---------------------------------------
cob = (val_mun.assign(con_dato=val_mun["mean"].notna())
       .groupby("dekad_id")
       .agg(municipios_con_dato=("con_dato", "sum"),
            pct_municipios=("con_dato", lambda s: round(100 * s.mean(), 1)),
            km2_validos=("km2", "sum"))
       .reset_index())
print("\ncobertura del panel municipal (ASI temporada primera):")
display(cob)


## 5. Capa de visualización

Todas las figuras del cuaderno se construyen con los ayudantes definidos aquí, para que colores, clases FAO, tooltips y notas de fuente sean idénticos en cada vista y no haya que repetir código.

Hay cinco constructores: `mapa_clases()` para coropletas con las clases oficiales, con animación por dekada opcional; `mapa_continuo()` para anomalías y cambios; `heatmap_panel()` para la matriz municipio × dekada, que muestra a la vez el cuándo y el dónde; `area_severidad()` para el área de cultivo en cada clase, que responde *cuánta superficie* y no solo *cuán intenso*; y `raster_detalle()` para la vista a resolución nativa de 1 km, que enseña lo que el promedio municipal esconde.

*Produce:* plantilla de estilo y funciones de figura. Sin salidas visuales.


In [ ]:
# Series nacionales oficiales de GIEWS, ponderadas por área de cultivo
REF_DEKAD, CASE_YEAR = "2019-09-D2", 2019

# El peso de cada departamento es su área de cultivo en píxeles válidos del
# ráster. Ponderar por número de departamentos le daría a Islas de la Bahía el
# mismo peso que a Olancho.
DEPT_WEIGHT = department_weights(municipal_series(SERIES["asi_gs1"],
                                                 [REF_DEKAD]))

# Serie nacional a partir del CSV departamental de GIEWS: es el dato de FAO, no
# un agregado propio, y es la referencia contra la cual se valida el panel.
_asi_csv = load_csv("asi_dekad_s1")
_asi_csv = _asi_csv[_asi_csv["Land_Type"].astype(str)
                    .str.contains("Crop", case=False, na=False)]
asi_history = national_from_csv(clip_period(_asi_csv), DEPT_WEIGHT)

print(f"serie nacional ASI · {len(asi_history):,} dekads · "
      f"{int(asi_history['Year'].min())}-{int(asi_history['Year'].max())}")


---

# Parte III. Caso 1 — Sequía agrícola, temporada primera 2019

## 6. Por qué 2019: contexto histórico del evento

Antes de mapear un evento hay que justificar por qué **es** el evento. Esta sección usa la serie oficial desde 2005 para situar la temporada primera de 2019 dentro de la historia reciente del índice, y solo después baja al detalle municipal.

Tres vistas, en orden de zoom. El ASI anual nacional por temporada muestra que 2019 es el máximo del periodo y que, contra lo que suele suponerse, el Niño de 2023-24 no produjo una sequía agrícola severa en Honduras. La climatología dekadal condensa el periodo completo en una sola matriz año × dekada, donde cada franja roja es una sequía. Y la comparación contra la normal muestra que 2019 se despega del rango habitual desde la segunda dekada de julio y no vuelve hasta octubre.

La vista climatológica se limita a la ventana de cultivo de la primera, de mayo a octubre, porque el ASI es acumulativo dentro de la temporada: fuera de ella el valor está congelado y graficarlo sugeriría una persistencia que no existe.

*Produce:* tres figuras de contexto y el panel municipal `ASI19`, guardado en caché, con el pico nacional identificado.


### 6.1 ¿Cuándo fue la última sequía agrícola severa?

Índice anual nacional por temporada desde 2005. Sitúa 2019 como el máximo del periodo y muestra que el Niño de 2023-24 no produjo una sequía agrícola severa en Honduras, contra lo que suele suponerse.

In [ ]:
# CELDA 6 · Caso 1 · eleccion del evento y contexto historico
# Explicacion, insumos y salidas: ver la seccion 6 del texto anterior.

# --- 6.1 ASI anual nacional por temporada (dato oficial GIEWS) ----------------
anual = pd.concat([load_csv("asi_annual_s1").assign(season="Primera (GS1)"),
                   load_csv("asi_annual_s2").assign(season="Postrera (GS2)")],
                  ignore_index=True)
anual["PROVINCE"] = anual["PROVINCE"].astype(str)
nac_anual = (anual[anual["PROVINCE"].str.upper().eq("ALL")]
             .dropna(subset=["DATA"])
             .rename(columns={"YEAR": "anio", "DATA": "asi"})
             .sort_values(["season", "anio"]))
ANIO_CASO = 2019

fig_anual = px.bar(nac_anual[nac_anual["anio"] >= START_YEAR], x="anio", y="asi",
                   color="season", barmode="group",
                   color_discrete_map={"Primera (GS1)": "#b0413e",
                                       "Postrera (GS2)": "#0b6fa4"},
                   labels={"anio": "", "asi": "ASI anual (% del area de cultivo)",
                           "season": ""},
                   hover_data={"asi": ":.2f"}, height=470)
fig_anual.add_vrect(x0=ANIO_CASO - 0.5, x1=ANIO_CASO + 0.5, line_width=0,
                    fillcolor="#f2c14e", opacity=0.35, layer="below")
fig_anual.add_annotation(x=ANIO_CASO, y=nac_anual["asi"].max(),
                         text="caso 1", showarrow=False, yshift=12,
                         font=dict(size=11, color="#8a6d1f"))
style_fig(fig_anual,
           "Cuando fue la ultima sequia agricola severa en Honduras",
           f"ASI nacional anual por temporada, {START_YEAR}-"
           f"{int(nac_anual[nac_anual['anio'] >= START_YEAR]['anio'].max())}. "
           "La primera 2019 es el maximo del periodo; el Nino 2023-24 NO produjo "
           "sequia agricola severa.", y_source=-0.22)
fig_anual.show()


### 6.2 Climatología dekadal del periodo

El periodo completo condensado en una matriz año × dekada, limitada a la ventana de cultivo de la primera. Cada franja roja horizontal es una sequía agrícola; la línea punteada marca 2019.

In [ ]:
# --- 6.2 Climatologia dekadal del periodo de analisis -------------------------
# La serie CSV es departamental; se pondera por area de cultivo real de cada
# departamento (pixeles validos del raster), no por numero de departamentos.
# El peso de cada departamento es su área de cultivo en píxeles válidos del
# ráster, no el número de departamentos.
PESO_DEP = department_weights(municipal_series(SERIES["asi_gs1"], [PEAK19]))

# Serie nacional oficial de GIEWS: es el dato de FAO, no un agregado propio.
_asi_csv = load_csv("asi_dekad_s1")
_asi_csv = _asi_csv[_asi_csv["Land_Type"].astype(str)
                    .str.contains("Crop", case=False, na=False)]
asi_hist = national_from_csv(clip_period(_asi_csv), PESO_DEP).rename(
    columns={"value": "valor", "dekad_of_year": "dek_anio"})
# El ASI es acumulativo dentro de la temporada: cuando la temporada termina el
# valor se congela hasta el reinicio, por eso la vista se limita a la ventana
# de cultivo de la primera (mayo-octubre, dekadas 13-30).
mat_hist = (asi_hist[asi_hist["dek_anio"].between(13, 30)]
            .pivot_table(index="Year", columns="dek_anio", values="valor"))
etq_dek = [f"{MONTH_ES[(k - 1) // 3 + 1]} D{(k - 1) % 3 + 1}" for k in mat_hist.columns]

fig_clim = go.Figure(go.Heatmap(
    z=mat_hist.values, x=etq_dek, y=mat_hist.index.astype(int),
    colorscale=SCALE_ASI, zmin=0, zmax=float(np.nanpercentile(mat_hist.values, 99.5)),
    xgap=0.5, ygap=0.5, colorbar=dict(title="ASI %", thickness=14, len=0.85),
    hovertemplate="%{y} · %{x}<br>ASI %{z:.1f}%<extra></extra>"))
fig_clim.add_hline(y=ANIO_CASO, line=dict(color="#111", width=1.4, dash="dot"))
fig_clim.update_layout(height=640, yaxis=dict(dtick=1, autorange="reversed"),
                       xaxis=dict(tickangle=-45))
style_fig(fig_clim, f"Climatologia del estres agricola, "
           f"{int(mat_hist.index.min())}-{int(mat_hist.index.max())}",
           "ASI nacional por dekada de la temporada primera (mayo-octubre), "
           "ponderado por area de cultivo de cada departamento. Cada franja roja "
           "es una sequia agricola; la linea punteada marca 2019.",
           y_source=-0.13, legend="off")
fig_clim.show()


### 6.3 2019 contra su propia normal

La franja azul es el rango habitual del índice para cada dekada. Lo que importa no es el nivel absoluto sino cuándo la curva de 2019 se sale de esa franja y cuánto tarda en volver.

In [ ]:
# --- 6.3 2019 contra su propia historia ---------------------------------------
per = asi_hist[asi_hist["dek_anio"].between(13, 30)]          # may a oct
base = per[per["Year"].between(START_YEAR, ANIO_CASO - 1)]
N_BASE = int(base["Year"].nunique())          # anios que sostienen la linea base
q = base.groupby("dek_anio")["valor"].quantile([.1, .5, .9]).unstack()
q.columns = ["p10", "p50", "p90"]
q = q.reset_index()
q["etq"] = [f"{MONTH_ES[(k - 1) // 3 + 1]} D{(k - 1) % 3 + 1}" for k in q["dek_anio"]]

fig_cmp = go.Figure()
fig_cmp.add_scatter(x=q["etq"], y=q["p90"],
                    name=f"p90 historico ({START_YEAR}-{ANIO_CASO - 1})",
                    line=dict(width=0, color="#c9d6e3"), showlegend=False)
fig_cmp.add_scatter(x=q["etq"], y=q["p10"],
                    name=f"rango p10-p90 ({START_YEAR}-{ANIO_CASO - 1})",
                    fill="tonexty", fillcolor="rgba(11,111,164,.15)",
                    line=dict(width=0, color="#c9d6e3"))
fig_cmp.add_scatter(x=q["etq"], y=q["p50"], name="mediana historica",
                    line=dict(color="#0b6fa4", width=2, dash="dot"))
for anio, color in ((2019, "#b0413e"), (2023, "#e07b39"), (2025, "#7a5195")):
    s = per[per["Year"] == anio]
    if len(s):
        fig_cmp.add_scatter(x=[f"{MONTH_ES[(k - 1) // 3 + 1]} D{(k - 1) % 3 + 1}"
                               for k in s["dek_anio"]], y=s["valor"],
                            name=str(anio), mode="lines+markers",
                            line=dict(color=color, width=2.6))
fig_cmp.update_layout(height=470, yaxis_title="ASI nacional (% del area de cultivo)")
style_fig(fig_cmp, "La temporada primera 2019 frente a la normal historica",
           f"Franja azul = rango habitual (p10-p90, {START_YEAR}-{ANIO_CASO - 1}, "
           f"n={N_BASE} anios). 2019 se despega desde la 2a dekada de julio y no "
           "vuelve al rango hasta octubre.",
           y_source=-0.24)
fig_cmp.show()


### 6.4 Panel municipal del evento

Descarga los rásteres de las dieciocho dekadas de la temporada, arma el panel municipio × dekada, identifica el pico nacional y lista los diez municipios más afectados en ese momento. Es la celda que alimenta toda la sección 7.

In [ ]:
# --- 6.4 Panel municipal del evento -------------------------------------------
DK_2019 = dekad_range(2019, 5, 1, 2019, 10, 3)
ASI19 = municipal_series(SERIES["asi_gs1"], DK_2019)
ASI19.to_parquet(CACHE / "panel_asi_2019.parquet")

nac19 = to_country(ASI19)
PICO19 = nac19.loc[nac19["mean"].idxmax(), "dekad_id"]
pico19 = ASI19[ASI19["dekad_id"] == PICO19]
alerta = nac19[nac19["mean"] > nac19["mean"].quantile(0.5)].iloc[0]["dekad_id"]

print(f"\nCASO 1 · sequia agricola temporada primera {ANIO_CASO}")
print(f"  dekadas analizadas   : {len(DK_2019)} ({DK_2019[0]} a {DK_2019[-1]})")
print(f"  pico nacional        : {dekad_label(PICO19)} · ASI {nac19['mean'].max():.1f}%")
print(f"  municipios con dato  : {pico19['mean'].notna().sum()} de {NZ}")
print(f"  area de cultivo con ASI>40 en el pico: "
      f"{pico19['km2_gt40'].sum():,.0f} km2 "
      f"({100 * pico19['km2_gt40'].sum() / pico19['km2'].sum():.1f}% del area de cultivo)")
print(f"  municipios con ASI>40: {(pico19['mean'] > 40).sum()} · "
      f"con ASI>70: {(pico19['mean'] > 70).sum()}")
display(pico19.nlargest(10, "mean")[["adm2_name", "adm1_name", "mean", "median",
                                     "p90", "pct_gt40", "km2"]]
        .rename(columns={"mean": "ASI medio"}).reset_index(drop=True))


## 7. Siete perspectivas del mismo evento

La misma sequía leída de siete maneras, porque cada pregunta operativa necesita una vista distinta y ninguna de ellas basta por sí sola.

Dónde y cuándo (mapa animado por dekada). Cómo escaló municipio por municipio, con la marca de la dekada en que ya había señal suficiente para una alerta temprana. Intensidad frente a impacto: el índice nacional resume lo primero, las barras de superficie afectada miden lo segundo, y no son lo mismo. Composición del territorio por clase de severidad. Dónde se concentra el daño en kilómetros cuadrados, que no coincide con dónde el promedio es más alto, porque los promedios altos suelen venir de municipios pequeños y los recursos se asignan por área. El detalle a 1 km, donde se ve que dentro de un mismo municipio conviven píxeles sin estrés y píxeles con ASI superior a 85. Y la trayectoria departamental en paneles comparables.

Una advertencia de lectura que la figura final explicita: el pico aislado de la primera dekada de mayo en Valle y Yoro es el valor residual de la temporada anterior antes del reinicio del índice, no un evento.

*Produce:* siete figuras interactivas.


In [ ]:
# CELDA 7 · Caso 1 · siete perspectivas del mismo evento
# Explicacion, insumos y salidas: ver la seccion 7 del texto anterior.
DEP19 = to_department(ASI19)
umbral_alerta = nac19[nac19["mean"] >= 10]
ALERTA19 = umbral_alerta.iloc[0]["dekad_id"] if len(umbral_alerta) else DK_2019[0]


### 7.1 Dónde y cuándo

Mapa animado por dekada con las clases oficiales de FAO. El foco del evento está en el oriente, Olancho y El Paraíso, y no en el Corredor Seco del suroeste, que es donde la intuición suele ubicarlo.

In [ ]:
# --- V1 · mapa animado de clases ---------------------------------------------
fig_v1 = class_map(ASI19, GEOJSON, "ASI", "Sequia agricola, temporada primera 2019 · ASI por municipio", "Cada cuadro es una dekada. El foco esta en el oriente (Olancho y El Paraiso) "
    "y no en el Corredor Seco del suroeste. Los municipios en blanco no tienen "
    "area de cultivo con dato en esa dekada.", hover_extra={"pct_gt40": ":.0f", "p90": ":.0f"})
fig_v1.show()


### 7.2 Cómo escaló, municipio por municipio

Matriz de los treinta municipios más afectados en el pico. La línea vertical marca la dekada en que el índice nacional superó el 10 %: desde ahí ya había señal suficiente para una alerta temprana.

In [ ]:
# --- V2 · escalada municipio x dekada ----------------------------------------
fig_v2 = heatmap_panel(
    ASI19, "mean",
    "Como escalo el estres, municipio por municipio",
    f"30 municipios mas afectados en el pico ({dekad_label(PICO19)}). "
    f"La linea marca la dekada en que el ASI nacional supero 10% "
    f"({dekad_label(ALERTA19)}): ahi ya habia senal para una alerta temprana.",
    family="ASI", top=30, ref_dekad=PICO19, label="ASI %")
fig_v2.add_vline(x=dekad_label(ALERTA19), line=dict(color="#111", width=1.6,
                                                    dash="dot"))
fig_v2.show()


### 7.3 Intensidad no es lo mismo que impacto

El índice nacional resume la intensidad; las barras miden la superficie de cultivo realmente afectada en cada dekada. Las dos curvas no tienen por qué moverse juntas.

In [ ]:
# --- V3 · curva nacional y superficie afectada -------------------------------
afect = (ASI19.groupby("dekad_id", as_index=False)
         .agg(km2_gt40=("km2_gt40", "sum"), km2_gt70=("km2_gt70", "sum"),
              km2=("km2", "sum")))
afect["pct_gt40"] = 100 * afect["km2_gt40"] / afect["km2"]
nac_plot = nac19.merge(afect, on="dekad_id", suffixes=("", "_a"))
etq = [dekad_label(c) for c in nac_plot["dekad_id"]]

fig_v3 = make_subplots(specs=[[{"secondary_y": True}]])
fig_v3.add_bar(x=etq, y=nac_plot["km2_gt40"], name="area de cultivo con ASI>40",
               marker_color="#e8a33d", opacity=0.85,
               hovertemplate="%{x}<br>%{y:,.0f} km2<extra></extra>",
               secondary_y=False)
fig_v3.add_bar(x=etq, y=nac_plot["km2_gt70"], name="area con ASI>70 (severo)",
               marker_color="#b0413e",
               hovertemplate="%{x}<br>%{y:,.0f} km2<extra></extra>",
               secondary_y=False)
fig_v3.add_scatter(x=etq, y=nac_plot["mean"], name="ASI nacional (%)",
                   mode="lines+markers", line=dict(color="#1f2430", width=2.6),
                   hovertemplate="%{x}<br>ASI %{y:.1f}%<extra></extra>",
                   secondary_y=True)
fig_v3.update_layout(height=480, barmode="overlay", xaxis=dict(tickangle=-45))
fig_v3.update_yaxes(title_text="km2 de area de cultivo", secondary_y=False)
fig_v3.update_yaxes(title_text="ASI nacional (%)", secondary_y=True,
                    showgrid=False)
style_fig(fig_v3, "Intensidad e impacto no son lo mismo",
           "El ASI nacional (linea) resume la intensidad; las barras muestran la "
           "superficie de cultivo realmente afectada en cada dekada.",
           y_source=-0.30)
fig_v3.show()


### 7.4 Composición del territorio por clase de severidad

Cuántos kilómetros cuadrados de cultivo hay en cada clase de FAO a lo largo de la temporada. Responde *cuánta superficie* y no solo *cuán intenso*.

In [ ]:
# --- V4 · composicion del territorio por clase de severidad ------------------
fig_v4 = severity_area_fig(
    ASI19, "ASI",
    "Composicion del area de cultivo por clase de severidad",
    "Area de cultivo de los municipios segun su clase FAO de ASI en cada dekada. "
    "En el pico, 1 de cada 5 km2 de cultivo del pais estaba en clase 40-55 o peor.")
fig_v4.show()


### 7.5 Dónde se concentra el daño en superficie

Los veinte municipios con más área afectada en el pico. No coincide con el ranking por promedio: los promedios altos suelen venir de municipios pequeños, y los recursos se asignan por área.

In [ ]:
# --- V5 · ranking por superficie afectada, no por promedio -------------------
top_km2 = (pico19.nlargest(20, "km2_gt40")
           .assign(etq=lambda d: d["adm2_name"] + " · " + d["adm1_name"])
           .sort_values("km2_gt40"))
fig_v5 = go.Figure()
fig_v5.add_bar(x=top_km2["km2_gt40"], y=top_km2["etq"], orientation="h",
               marker=dict(color=top_km2["mean"], colorscale=SCALE_ASI,
                           cmin=0, cmax=100,
                           colorbar=dict(title="ASI medio<br>del municipio",
                                         thickness=14, len=0.8)),
               customdata=np.stack([top_km2["mean"], top_km2["km2"],
                                    top_km2["pct_gt40"]], axis=-1),
               hovertemplate=("%{y}<br>area afectada %{x:,.0f} km2"
                              "<br>ASI medio %{customdata[0]:.1f}%"
                              "<br>area de cultivo %{customdata[1]:,.0f} km2"
                              "<br>%{customdata[2]:.0f}% del cultivo del municipio"
                              "<extra></extra>"))
fig_v5.update_layout(height=680, xaxis_title="km2 de cultivo con ASI>40 en el pico",
                     margin=dict(l=230))
style_fig(fig_v5, "Donde se concentra el dano en superficie",
           f"20 municipios con mas km2 de cultivo afectado en {dekad_label(PICO19)}. "
           "El promedio municipal alto suele venir de municipios chicos: para "
           "asignar recursos importa el area, no solo el indice.",
           y_source=-0.14, legend="off")
fig_v5.show()


### 7.6 El mismo pico a 1 km de resolución

Lo que el promedio municipal esconde. Dentro de un mismo municipio conviven píxeles sin estrés y píxeles por encima de 85.

In [ ]:
# --- V6 · detalle a resolucion nativa ----------------------------------------
fig_v6 = raster_detail(
    export_tif("ASI_D", raster_name("ASI_D", PICO19, SUF_GS1)),
    "El mismo pico a 1 km de resolucion",
    f"{dekad_label(PICO19)}. El promedio municipal suaviza: dentro de un mismo "
    "municipio conviven pixeles sin estres y pixeles con ASI>85. En gris, "
    "superficie sin area de cultivo o fuera de temporada.",
    family="ASI", label="ASI %")
fig_v6.show()


### 7.7 Trayectoria departamental

Paneles con la misma escala para comparar departamentos. El pico aislado de la primera dekada de mayo en Valle y Yoro es el valor residual de la temporada anterior antes del reinicio del índice, no un evento.

In [ ]:
# --- V7 · trayectoria por departamento (small multiples) ---------------------
fig_v7 = px.line(DEP19.sort_values("dekad_id"), x="dekad_id", y="mean",
                 facet_col="adm1_name", facet_col_wrap=5, markers=True,
                 height=760, labels={"mean": "ASI (%)", "dekad_id": ""},
                 color_discrete_sequence=["#b0413e"])
fig_v7.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1],
                                              font=dict(size=11)))
fig_v7.update_xaxes(tickangle=-60, tickfont=dict(size=8))
fig_v7.add_hline(y=40, line=dict(color="#888", width=1, dash="dot"))
style_fig(fig_v7, "Trayectoria departamental de la temporada primera 2019",
           "Misma escala en todos los paneles; la linea punteada es el umbral "
           "ASI=40. Olancho, El Paraiso y Francisco Morazan concentran el evento. "
           "El pico aislado de la 1a dekada de mayo en Valle y Yoro es el valor "
           "residual de la temporada anterior antes del reinicio del indice, no un "
           "evento: el ASI se acumula dentro de la temporada y se reinicia con ella.",
           y_source=-0.10, legend="off")
fig_v7.show()

print("caso 1 · 7 figuras generadas ·",
      f"panel {ASI19.shape[0]:,} filas x {ASI19.shape[1]} columnas")


---

# Parte IV. Caso 2 — Exceso de lluvia: huracanes Eta e Iota

## 8. Lluvia, el punto ciego del ASI, y el deterioro vegetal

Eta e Iota tocaron Honduras con dos semanas de diferencia en noviembre de 2020. El caso está aquí por una razón metodológica: **el ASI mide déficit hídrico y por construcción no puede ver un exceso de agua**. Mostrar ese límite explícitamente vale más que esconderlo.

La sección sigue el evento con la anomalía de lluvia y con el VCI, y documenta el punto ciego: con la anomalía de lluvia más alta de la serie, el ASI de la postrera se movió dentro de su rango habitual y el VCI nacional incluso subió, porque más agua reverdece el promedio del país. El daño por exceso es local y solo aparece al bajar a municipio.

*Produce:* serie de lluvia observada contra su promedio histórico, la comparación ASI–VCI que evidencia el punto ciego, y el mapa animado del deterioro vegetal entre octubre de 2020 y febrero de 2021.


In [ ]:
# CELDA 8 · Caso 2 · precipitacion extrema: Eta e Iota (nov 2020)
# Explicacion, insumos y salidas: ver la seccion 8 del texto anterior.
DK_ETA = dekad_range(2020, 10, 1, 2021, 2, 3)          # 15 dekadas
VCI20  = municipal_series(SERIES["vci"], DK_ETA)
ASI20  = municipal_series(SERIES["asi_gs2"], dekad_range(2020, 9, 1, 2021, 1, 3))
VCI20.to_parquet(CACHE / "panel_vci_2020.parquet")
ASI20.to_parquet(CACHE / "panel_asi_2020_gs2.parquet")

ANTES, IMPACTO, RECUP = "2020-10-D3", "2020-11-D2", "2021-02-D3"
nac_vci = to_country(VCI20)
nac_asi20 = to_country(ASI20)


### 8.1 La lluvia observada contra su promedio histórico

Lluvia dekadal nacional ponderada por área de cultivo, de agosto de 2020 a febrero de 2021, contra la referencia de largo plazo que publica FAO. Las dos dekadas de noviembre quedan marcadas.

In [ ]:
# --- 8.1 Lluvia nacional: observada vs promedio historico ---------------------
# Lluvia dekadal nacional y su promedio de largo plazo, ponderando
# departamentos por área de cultivo. La LTA es la que publica FAO y NO se
# recalcula: es la referencia oficial contra la que compara GIEWS.
lluvia = national_from_csv(clip_period(load_csv("rain_dekad")), PESO_DEP,
                           lta_col="Data_long_term_Average").rename(
    columns={"value": "obs"})
vent = lluvia[lluvia["dekad_id"].isin(dekad_range(2020, 8, 1, 2021, 2, 3))].copy()
vent["etq"] = [dekad_label(c) for c in vent["dekad_id"]]

fig_w1 = make_subplots(specs=[[{"secondary_y": True}]])
fig_w1.add_bar(x=vent["etq"], y=vent["obs"], name="lluvia observada",
               marker_color="#3b7dd8", opacity=0.9,
               hovertemplate="%{x}<br>%{y:.0f} mm<extra></extra>")
fig_w1.add_scatter(x=vent["etq"], y=vent["lta"], name="promedio historico (LTA)",
                   mode="lines+markers", line=dict(color="#0b3d91", dash="dot"),
                   hovertemplate="%{x}<br>LTA %{y:.0f} mm<extra></extra>")
fig_w1.add_scatter(x=vent["etq"], y=vent["anom_pct"], name="anomalia (%)",
                   mode="lines", line=dict(color="#b0413e", width=2),
                   secondary_y=True,
                   hovertemplate="%{x}<br>anomalia %{y:+.0f}%<extra></extra>")
fig_w1.add_vrect(x0="1a dek nov 2020", x1="2a dek nov 2020", line_width=0,
                 fillcolor="#b0413e", opacity=0.10, layer="below")
fig_w1.add_annotation(x="2a dek nov 2020", y=1.02, yref="paper", showarrow=False,
                      text="Eta (3-5 nov) e Iota (16-18 nov)",
                      font=dict(size=11, color="#b0413e"))
fig_w1.update_yaxes(title_text="mm por dekada", secondary_y=False)
fig_w1.update_yaxes(title_text="anomalia sobre la LTA (%)", secondary_y=True,
                    showgrid=False)
fig_w1.update_layout(height=470, xaxis=dict(tickangle=-45))
style_fig(fig_w1, "Eta e Iota en la serie de lluvia de FAO GIEWS",
           "Lluvia dekadal nacional ponderada por area de cultivo, agosto 2020 a "
           "febrero 2021, contra el promedio historico 1989-2015.", y_source=-0.30)
fig_w1.show()

pico_lluvia = vent.loc[vent["anom_pct"].idxmax()]
print(f"maxima anomalia de lluvia: {dekad_label(pico_lluvia['dekad_id'])} · "
      f"{pico_lluvia['obs']:.0f} mm vs {pico_lluvia['lta']:.0f} mm de LTA "
      f"({pico_lluvia['anom_pct']:+.0f}%)")


### 8.2 El punto ciego del ASI

Con la anomalía de lluvia más alta de la serie, el índice de estrés de la postrera se movió dentro de su rango habitual y el de vegetación incluso subió, porque más agua reverdece el promedio del país. Ningún índice de sequía sirve para medir una inundación.

In [ ]:
# --- 8.2 El punto ciego del ASI ----------------------------------------------
comun = sorted(set(nac_vci["dekad_id"]) & set(nac_asi20["dekad_id"]))
cmp2 = (nac_vci[nac_vci["dekad_id"].isin(comun)][["dekad_id", "mean"]]
        .rename(columns={"mean": "vci"})
        .merge(nac_asi20[["dekad_id", "mean"]].rename(columns={"mean": "asi"}),
               on="dekad_id")
        .merge(lluvia[["dekad_id", "anom_pct"]], on="dekad_id", how="left"))
cmp2["etq"] = [dekad_label(c) for c in cmp2["dekad_id"]]

fig_w2 = make_subplots(specs=[[{"secondary_y": True}]])
fig_w2.add_scatter(x=cmp2["etq"], y=cmp2["vci"], name="VCI nacional (0-1)",
                   mode="lines+markers", line=dict(color="#2f8f4e", width=2.8),
                   hovertemplate="%{x}<br>VCI %{y:.3f}<extra></extra>")
fig_w2.add_scatter(x=cmp2["etq"], y=cmp2["asi"], name="ASI postrera (%)",
                   mode="lines+markers", line=dict(color="#c9a227", width=2.4,
                                                   dash="dash"),
                   secondary_y=True,
                   hovertemplate="%{x}<br>ASI %{y:.2f}%<extra></extra>")
fig_w2.add_hline(y=0.35, line=dict(color="#ff8900", width=1.2, dash="dot"))
fig_w2.add_annotation(x=cmp2["etq"].iloc[0], y=0.35, text="umbral FAO VCI 0,35",
                      showarrow=False, yshift=10, xanchor="left",
                      font=dict(size=10, color="#ff8900"))
fig_w2.update_yaxes(title_text="VCI (0-1)", secondary_y=False, range=[0, 1])
fig_w2.update_yaxes(title_text="ASI temporada postrera (%)", secondary_y=True,
                    showgrid=False)
fig_w2.update_layout(height=460, xaxis=dict(tickangle=-45))
_v_antes = nac_vci.set_index("dekad_id").loc[ANTES, "mean"]
_v_imp = nac_vci.set_index("dekad_id").loc[IMPACTO, "mean"]
style_fig(fig_w2, "Ningun indice de sequia sirve para medir una inundacion",
           f"Con la anomalia de lluvia mas alta de la serie, el ASI de la postrera se "
           f"movio dentro de su rango habitual (maximo {cmp2['asi'].max():.1f}%) y el "
           f"VCI nacional incluso subio de {_v_antes:.2f} a {_v_imp:.2f}: mas agua "
           "reverdece el promedio del pais. El dano por exceso es local, hay que "
           "buscarlo por municipio y no en el agregado nacional.",
           y_source=-0.30)
fig_w2.show()


### 8.3 El deterioro vegetal, dekada a dekada

Mapa animado de octubre de 2020 a febrero de 2021. El promedio nacional no se mueve, pero un grupo de municipios del valle de Sula, Santa Bárbara y el occidente cae con fuerza en la segunda dekada de noviembre.

In [ ]:
# --- 8.3 Mapa animado del deterioro vegetal ----------------------------------
fig_w3 = class_map(VCI20, GEOJSON, "VCI", "Huracanes Eta e Iota · condicion de la vegetacion por municipio", "VCI dekadal de octubre 2020 a febrero 2021. Rojo = vegetacion en peor estado "
    "que su historia reciente; verde = mejor. El promedio nacional no se mueve, pero "
    "un grupo de municipios del valle de Sula, Santa Barbara y el occidente cae con "
    "fuerza en la 2a dekada de noviembre.", hover_extra={"pct_lt0.35": ":.0f"})
fig_w3.show()

print(f"\nCASO 2 · Eta e Iota · dekadas {DK_ETA[0]} a {DK_ETA[-1]}")
print(f"  VCI nacional antes ({dekad_label(ANTES)}): "
      f"{nac_vci.set_index('dekad_id').loc[ANTES, 'mean']:.3f}")
print(f"  VCI nacional impacto ({dekad_label(IMPACTO)}): "
      f"{nac_vci.set_index('dekad_id').loc[IMPACTO, 'mean']:.3f}")
print(f"  municipios bajo VCI 0.35 en el impacto: "
      f"{(VCI20[VCI20['dekad_id'] == IMPACTO]['mean'] < 0.35).sum()} de {NZ}")
print("  lectura: el promedio nacional del VCI no cae (mas agua reverdece); el dano")
print("  por exceso solo aparece al bajar a municipio y a las llanuras de inundacion.")


## 9. Cinco lecturas municipales adicionales

Quién cayó más entre la dekada previa y la del impacto. Cómo se propagó y cuánto persistió, municipio por municipio. La relación entre lluvia y daño a escala departamental, que resulta **prácticamente nula**: es un resultado negativo útil, porque indica que el daño por inundación lo decide la topografía y el drenaje —las llanuras del Ulúa y el Chamelecón— y no el total de lluvia del departamento. El detalle a 1 km, donde ese alineamiento con las llanuras de inundación se ve directamente. Y el saldo tres meses después, que separa a los municipios que ya habían vuelto a su condición previa de los que seguían por debajo.

*Produce:* cinco figuras y la tabla `delta` con el antes, el impacto y la recuperación por municipio.


In [ ]:
# CELDA 9 · Caso 2 · cinco lecturas municipales adicionales
# Explicacion, insumos y salidas: ver la seccion 9 del texto anterior.
antes_m   = VCI20[VCI20["dekad_id"] == ANTES].set_index("adm2_code")
impacto_m = VCI20[VCI20["dekad_id"] == IMPACTO].set_index("adm2_code")
recup_m   = VCI20[VCI20["dekad_id"] == RECUP].set_index("adm2_code")

delta = (antes_m[["adm2_name", "adm1_name", "km2", "mean"]]
         .rename(columns={"mean": "antes"})
         .join(impacto_m["mean"].rename("impacto"))
         .join(recup_m["mean"].rename("recuperacion"))
         .dropna(subset=["antes", "impacto"]))
delta["caida"] = delta["antes"] - delta["impacto"]
delta["saldo"] = delta["recuperacion"] - delta["antes"]
delta = delta.reset_index()


### 9.1 Quién cayó más en una sola dekada

Los veinticinco municipios con mayor caída entre la dekada previa y la del impacto. El punto verde es el antes, el rojo el después; la línea vertical es el umbral de alerta de FAO.

In [ ]:
# --- W4 · quien cayo mas (dumbbell) -------------------------------------------
sel = delta.nlargest(25, "caida").sort_values("caida")
sel["etq"] = sel["adm2_name"] + " · " + sel["adm1_name"]
fig_w4 = go.Figure()
for _, f in sel.iterrows():
    fig_w4.add_scatter(x=[f["antes"], f["impacto"]], y=[f["etq"]] * 2,
                       mode="lines", line=dict(color="#c8ccd4", width=3),
                       showlegend=False, hoverinfo="skip")
fig_w4.add_scatter(x=sel["antes"], y=sel["etq"], mode="markers",
                   name=f"antes · {dekad_label(ANTES)}",
                   marker=dict(size=11, color="#2f8f4e"),
                   hovertemplate="%{y}<br>VCI %{x:.3f}<extra></extra>")
fig_w4.add_scatter(x=sel["impacto"], y=sel["etq"], mode="markers",
                   name=f"tras Eta · {dekad_label(IMPACTO)}",
                   marker=dict(size=11, color="#b0413e"),
                   customdata=np.stack([sel["caida"], sel["km2"]], axis=-1),
                   hovertemplate="%{y}<br>VCI %{x:.3f}<br>caida "
                                 "%{customdata[0]:.3f}<br>%{customdata[1]:,.0f} km2"
                                 "<extra></extra>")
fig_w4.add_vline(x=0.35, line=dict(color="#ff8900", dash="dash"))
fig_w4.add_annotation(x=0.35, y=1.02, yref="paper", showarrow=False,
                      text="umbral FAO 0,35", font=dict(size=11, color="#ff8900"))
fig_w4.update_layout(height=720, margin=dict(l=240),
                     xaxis_title="Indice de Condicion de la Vegetacion (0-1)")
style_fig(fig_w4, "Deterioro de la vegetacion en una sola dekada",
           "25 municipios con mayor caida de VCI entre la 3a dekada de octubre y "
           "la 2a de noviembre de 2020. Predominan Santa Barbara y Cortes (valle "
           "de Sula) junto con el occidente montanoso: Copan, Ocotepeque, Lempira e "
           "Intibuca.", y_source=-0.13)
fig_w4.show()


### 9.2 Propagación y persistencia

Los treinta municipios con peor condición en la dekada de impacto, seguidos en el tiempo. Se ve el golpe de noviembre y cuánto tarda cada uno en volver al verde.

In [ ]:
# --- W5 · propagacion municipio x dekada --------------------------------------
fig_w5 = heatmap_panel(
    VCI20, "mean",
    "Propagacion y persistencia del dano",
    "30 municipios con menor VCI en la dekada de impacto. Se ve el golpe de "
    "noviembre y cuanto tarda cada municipio en volver al verde.",
    family="VCI", top=30, ref_dekad=IMPACTO, value_range=(0, 1), label="VCI")
fig_w5.show()


### 9.3 Lluvia y daño no se corresponden a escala departamental

Correlación prácticamente nula. Es un resultado negativo útil: el daño por inundación lo decide la topografía y el drenaje —las llanuras del Ulúa y el Chamelecón— y no el total de lluvia del departamento.

In [ ]:
# --- W6 · relacion lluvia-dano por departamento -------------------------------
r_nov = load_csv("rain_dekad")
r_nov = r_nov[r_nov["dekad_id"].isin(dekad_range(2020, 11, 1, 2020, 11, 3))]
r_nov = (r_nov.groupby(["ADM1_CODE", "Province"], as_index=False)
         [["Data", "Data_long_term_Average"]].sum())
r_nov["anom_pct"] = 100 * (r_nov["Data"] / r_nov["Data_long_term_Average"] - 1)
r_nov["adm1_code"] = r_nov["ADM1_CODE"].astype("Int64").astype(str)

vci_dep = to_department(VCI20)
dep_cmp = (vci_dep[vci_dep["dekad_id"] == ANTES][["adm1_code", "adm1_name", "mean", "km2"]]
           .rename(columns={"mean": "antes"})
           .merge(vci_dep[vci_dep["dekad_id"] == IMPACTO][["adm1_code", "mean"]]
                  .rename(columns={"mean": "impacto"}), on="adm1_code")
           .merge(r_nov[["adm1_code", "anom_pct", "Data"]], on="adm1_code"))
dep_cmp["caida"] = dep_cmp["antes"] - dep_cmp["impacto"]
rr = np.corrcoef(dep_cmp["anom_pct"], dep_cmp["caida"])[0, 1]

fig_w6 = px.scatter(dep_cmp, x="anom_pct", y="caida", size="km2", color="adm1_name",
                    text="adm1_name", size_max=42, height=560,
                    labels={"anom_pct": "anomalia de lluvia de noviembre 2020 (%)",
                            "caida": "caida del VCI (puntos del indice)",
                            "adm1_name": ""},
                    hover_data={"Data": ":.0f", "antes": ":.2f", "impacto": ":.2f",
                                "km2": ":,.0f"})
fig_w6.update_traces(textposition="top center", textfont=dict(size=9))
style_fig(fig_w6, "A escala departamental la relacion lluvia-dano no aparece",
           f"Cada burbuja es un departamento; el tamano es su area de cultivo. La "
           f"correlacion entre la anomalia de lluvia de noviembre y la caida del VCI es "
           f"r={rr:.2f}, practicamente nula. Es un resultado negativo util: el dano por "
           "inundacion lo decide la topografia y el drenaje (llanuras del Ulua y el "
           "Chamelecon), no el total de lluvia del departamento.",
           y_source=-0.16, legend="off")
fig_w6.show()


### 9.4 El impacto a 1 km

A resolución nativa el daño se alinea con las llanuras de inundación, un patrón que el promedio municipal diluye.

In [ ]:
# --- W7 · detalle 1 km del impacto --------------------------------------------
fig_w7 = raster_detail(
    export_tif("VCI_D", raster_name("VCI_D", IMPACTO)),
    "Condicion de la vegetacion a 1 km tras Eta",
    f"{dekad_label(IMPACTO)}. El dano se alinea con las llanuras de inundacion "
    "del Ulua y el Chamelecon, un patron que el promedio municipal diluye.",
    family="VCI", label="VCI")
fig_w7.show()


### 9.5 El saldo tres meses después

Diferencia entre la condición de febrero de 2021 y la previa al evento. Rojo: el municipio sigue por debajo de donde estaba; azul: ya lo superó.

In [ ]:
# --- W8 · quien no se recupero ------------------------------------------------
fig_w8 = continuous_map(delta.dropna(subset=["saldo"]), GEOJSON, "saldo", "Saldo tres meses despues del evento", f"VCI de {dekad_label(RECUP)} menos VCI de {dekad_label(ANTES)}. Rojo: el "
    "municipio sigue por debajo de su condicion previa; azul: ya la supero.", scale="RdBu", value_range=(-0.35, 0.35), bar_label="Δ VCI", hover_extra={"antes": ":.2f", "impacto": ":.2f", "recuperacion": ":.2f"})
fig_w8.show()

n_peor = int((delta["saldo"] < -0.05).sum())
print(f"caso 2 · 5 figuras adicionales · {n_peor} municipios seguian por debajo "
      f"de su VCI previo en {dekad_label(RECUP)} "
      f"({100 * n_peor / len(delta):.0f}% de los municipios con dato)")


---

# Parte V. Estado actual

## 10. Últimos 18 meses publicados

La ventana **no se fija a mano**: se le pregunta al catálogo cuál es la última dekada efectivamente publicada y se retroceden 54 dekadas. Así esta parte sigue siendo válida cuando FAO publique rásteres nuevos, sin editar nada.

Se descargan las dos temporadas, primera y postrera, porque el ASI solo tiene sentido dentro de su ventana de cultivo; fuera de ella el ráster trae la bandera de fuera de temporada y el municipio queda sin dato. En las dekadas en que ambas temporadas están activas se conserva el **peor caso vigente**, criterio conservador para alerta temprana, dejando registrado de cuál temporada proviene. Se trae además el VCI, que sí es continuo todo el año y sirve de contraste.

Esta es la celda más pesada del cuaderno en la primera ejecución.

*Produce:* los paneles `ASI18` y `vci_18m` guardados en caché, el resumen numérico del estado actual, la foto de la última dekada publicada y la película de los 18 meses.


In [ ]:
# CELDA 10 · Estado actual: ultimos 18 meses disponibles
# Explicacion, insumos y salidas: ver la seccion 10 del texto anterior.
ULTIMA = last_dekad("ASI_D", season="GS1", landcover="C")
DK_18M = dekad_window(ULTIMA, 54)
print(f"ventana: {DK_18M[0]} a {DK_18M[-1]} ({len(DK_18M)} dekadas · "
      f"{dekad_label(DK_18M[0])} - {dekad_label(DK_18M[-1])})\n")

asi_gs1 = municipal_series(SERIES["asi_gs1"], DK_18M)
asi_gs2 = municipal_series(SERIES["asi_gs2"], DK_18M)
vci_18m = municipal_series(SERIES["vci"], DK_18M)

# --- Union de temporadas: se conserva el peor caso vigente --------------------
# En las dekadas en que las dos temporadas estan activas se toma el maximo ASI
# (criterio conservador para alerta temprana) y se deja registrado de cual viene.
ASI18 = (pd.concat([asi_gs1, asi_gs2], ignore_index=True)
         .dropna(subset=["mean"])
         .sort_values(["adm2_code", "dekad_id", "mean"], ascending=[True, True, False])
         .drop_duplicates(["adm2_code", "dekad_id"], keep="first")
         .reset_index(drop=True))
ASI18["clase"] = classify(ASI18["mean"], "ASI").astype(str)
ASI18.to_parquet(CACHE / "panel_asi_18m.parquet")
vci_18m.to_parquet(CACHE / "panel_vci_18m.parquet")

nac18 = to_country(ASI18)
nac18_vci = to_country(vci_18m)
dep18 = to_department(ASI18)
ACTUAL = ASI18[ASI18["dekad_id"] == ULTIMA].copy()
ACTUAL_VCI = vci_18m[vci_18m["dekad_id"] == ULTIMA]

cobertura = (ASI18.groupby("dekad_id")
             .agg(municipios=("mean", "count"),
                  temporadas=("season", lambda s: "+".join(sorted(set(s.dropna())))),
                  km2=("km2", "sum"))
             .reset_index())

print(f"panel ASI 18 meses : {len(ASI18):,} filas · "
      f"{ASI18['dekad_id'].nunique()} dekadas · {ASI18['adm2_code'].nunique()} municipios")
print(f"panel VCI 18 meses : {len(vci_18m):,} filas")
print(f"\nESTADO EN LA ULTIMA DEKADA PUBLICADA · {dekad_label(ULTIMA)}")
print(f"  ASI nacional          : {nac18[nac18['dekad_id'] == ULTIMA]['mean'].iloc[0]:.2f}% "
      f"(media de las 54 dekadas: {nac18['mean'].mean():.2f}%)")
print(f"  VCI nacional          : {nac18_vci[nac18_vci['dekad_id'] == ULTIMA]['mean'].iloc[0]:.3f}")
print(f"  municipios con dato   : {ACTUAL['mean'].notna().sum()} de {NZ}")
print(f"  municipios ASI > 30   : {(ACTUAL['mean'] > 30).sum()} · "
      f"> 40: {(ACTUAL['mean'] > 40).sum()} · > 70: {(ACTUAL['mean'] > 70).sum()}")
print(f"  area de cultivo ASI>40: {ACTUAL['km2_gt40'].sum():,.0f} km2 "
      f"({100 * ACTUAL['km2_gt40'].sum() / ACTUAL['km2'].sum():.1f}% del area con dato)")
print(f"  municipios VCI < 0.35 : {(ACTUAL_VCI['mean'] < 0.35).sum()}")
display(cobertura.tail(8))


### 10.1 La foto del momento

Estado municipal en la última dekada publicada por FAO. Verde: sin estrés por déficit hídrico. Blanco: municipio fuera de temporada o sin área de cultivo, que no es lo mismo que sin estrés.

In [ ]:
# --- A1 · foto del momento -----------------------------------------------------
fig_a1 = class_map(ACTUAL, GEOJSON, "ASI", f"Estado agroclimatico de Honduras · {dekad_label(ULTIMA)}", "ASI municipal en la ultima dekada publicada por FAO GIEWS. Verde: sin "
    "estres por deficit hidrico; rojo: mas del 70% del area de cultivo del "
    "municipio bajo estres. En blanco, municipios fuera de temporada o sin cultivo.", animation=None, hover_extra={"pct_gt40": ":.0f", "p90": ":.0f"})
fig_a1.show()


### 10.2 Los últimos 18 meses, dekada a dekada

La misma vista animada sobre toda la ventana. En cada momento se muestra la temporada activa, primera o postrera.

In [ ]:
# --- A2 · la pelicula de los 18 meses ------------------------------------------
fig_a2 = class_map(ASI18, GEOJSON, "ASI", "Los ultimos 18 meses, dekada a dekada", f"{dekad_label(DK_18M[0])} a {dekad_label(ULTIMA)}. Se muestra la temporada "
    "activa en cada momento (primera o postrera); los municipios en blanco estan "
    "fuera de su ventana de cultivo.", hover_extra={"pct_gt40": ":.0f", "season": True})
fig_a2.show()


## 11. Cinco lecturas de la ventana reciente

Un tablero con estrés hídrico, condición de la vegetación y anomalía de lluvia sobre el mismo eje temporal, que permite ver si los tres indicadores cuentan la misma historia. La persistencia municipio por municipio, donde los huecos son dekadas fuera de temporada y no ceros. La foto del momento contra el propio promedio de cada municipio en la ventana, que separa el deterioro reciente del estrés crónico. La dispersión entre municipios en cada dekada, porque un promedio nacional bajo puede convivir con una cola de municipios en crisis. Y el mapa acumulado del periodo, que identifica cronicidad en lugar de episodios.

*Produce:* cinco figuras y la tabla `persist` con la persistencia por municipio.


In [ ]:
# CELDA 11 · Estado actual · cinco lecturas de la ventana reciente
# Explicacion, insumos y salidas: ver la seccion 11 del texto anterior.
lluvia18 = lluvia[lluvia["dekad_id"].isin(DK_18M)].copy()
lluvia18["date"] = lluvia18["dekad_id"].map(dekad_date)


### 11.1 Tablero: estrés, vegetación y lluvia en el mismo eje

Tres indicadores independientes sobre el mismo eje temporal. Lo que se busca es si cuentan la misma historia o se contradicen.

In [ ]:
# --- A3 · tablero: estres, vegetacion y lluvia en el mismo eje de tiempo -------
fig_a3 = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=("ASI nacional · % del area de cultivo bajo estres hidrico",
                    "VCI nacional · condicion de la vegetacion (0-1)",
                    "Anomalia de lluvia sobre el promedio historico (%)"))
fig_a3.add_scatter(x=nac18["date"], y=nac18["mean"], name="ASI",
                   mode="lines+markers", line=dict(color="#b0413e", width=2.4),
                   fill="tozeroy", fillcolor="rgba(176,65,62,.18)",
                   hovertemplate="%{x|%d %b %Y}<br>ASI %{y:.1f}%<extra></extra>",
                   row=1, col=1)
fig_a3.add_scatter(x=nac18_vci["date"], y=nac18_vci["mean"], name="VCI",
                   mode="lines", line=dict(color="#2f8f4e", width=2.4),
                   hovertemplate="%{x|%d %b %Y}<br>VCI %{y:.3f}<extra></extra>",
                   row=2, col=1)
fig_a3.add_hline(y=0.35, line=dict(color="#ff8900", width=1.1, dash="dot"),
                 row=2, col=1)
fig_a3.add_bar(x=lluvia18["date"], y=lluvia18["anom_pct"], name="anomalia lluvia",
               marker_color=np.where(lluvia18["anom_pct"] >= 0, "#3b7dd8", "#c98b3b"),
               hovertemplate="%{x|%d %b %Y}<br>%{y:+.0f}%<extra></extra>",
               row=3, col=1)
fig_a3.add_hline(y=0, line=dict(color="#666", width=1), row=3, col=1)
fig_a3.update_layout(height=760, showlegend=False)
fig_a3.update_annotations(font_size=12, x=0, xanchor="left")
style_fig(fig_a3, f"Honduras en los ultimos 18 meses · hasta {dekad_label(ULTIMA)}",
           "Tres indicadores independientes sobre el mismo eje temporal: deficit "
           "hidrico acumulado en el cultivo (ASI), estado de la vegetacion (VCI) y "
           "lluvia observada frente a su normal.", y_source=-0.11, legend="off")
fig_a3.show()


### 11.2 Persistencia municipio por municipio

Los veinticinco municipios con mayor estrés promedio en la ventana. Los huecos son dekadas fuera de temporada: el índice no existe todo el año y eso hay que decirlo, no rellenarlo con ceros.

In [ ]:
# --- A4 · persistencia: quien estuvo mal y por cuanto tiempo -------------------
persist = (ASI18.assign(estres=ASI18["mean"] > 40)
           .groupby(["adm2_code", "adm2_name", "adm1_name"], as_index=False)
           .agg(dek_con_dato=("mean", "count"), dek_estres=("estres", "sum"),
                asi_medio=("mean", "mean"), asi_max=("mean", "max"),
                km2=("km2", "mean")))
persist["pct_dek_estres"] = 100 * persist["dek_estres"] / persist["dek_con_dato"]

top_pers = persist.nlargest(25, "asi_medio")
mat = (ASI18[ASI18["adm2_code"].isin(top_pers["adm2_code"])]
       .assign(etq=lambda d: d["adm2_name"] + " · " + d["adm1_name"])
       .pivot_table(index="etq", columns="dekad_id", values="mean")
       .reindex(columns=[d for d in DK_18M if d in set(ASI18["dekad_id"])]))
mat = mat.loc[mat.mean(axis=1).sort_values().index]

fig_a4 = go.Figure(go.Heatmap(
    z=mat.values, x=[dekad_label(c) for c in mat.columns], y=mat.index,
    colorscale=SCALE_ASI, zmin=0, zmax=100, xgap=0.5, ygap=1,
    colorbar=dict(title="ASI %", thickness=14, len=0.85),
    hovertemplate="%{y}<br>%{x}<br>ASI %{z:.1f}%<extra></extra>"))
fig_a4.update_layout(height=740, margin=dict(l=240),
                     xaxis=dict(tickangle=-60, tickfont=dict(size=8)),
                     yaxis=dict(tickfont=dict(size=10)))
style_fig(fig_a4, "Persistencia del estres municipio por municipio",
           "25 municipios con mayor ASI promedio en la ventana. Los huecos son "
           "dekadas fuera de temporada: el ASI no existe todo el ano y eso hay que "
           "decirlo, no rellenarlo con ceros.", y_source=-0.15, legend="off")
fig_a4.show()


### 11.3 Lo de hoy contra el propio historial de cada municipio

Punto rojo: la última dekada publicada. Punto gris: el promedio del municipio en los dieciocho meses. Cuando el rojo está muy a la derecha del gris, el deterioro es reciente y no crónico.

In [ ]:
# --- A5 · foto del momento contra el propio historial de 18 meses -------------
rank = (ACTUAL.merge(persist[["adm2_code", "asi_medio", "pct_dek_estres"]],
                     on="adm2_code")
        .nlargest(20, "mean")
        .assign(etq=lambda d: d["adm2_name"] + " · " + d["adm1_name"])
        .sort_values("mean"))
fig_a5 = go.Figure()
for _, f in rank.iterrows():
    fig_a5.add_scatter(x=[f["asi_medio"], f["mean"]], y=[f["etq"]] * 2,
                       mode="lines", line=dict(color="#c8ccd4", width=3),
                       showlegend=False, hoverinfo="skip")
fig_a5.add_scatter(x=rank["asi_medio"], y=rank["etq"], mode="markers",
                   name="promedio de los 18 meses",
                   marker=dict(size=10, color="#8b93a1"),
                   hovertemplate="%{y}<br>promedio %{x:.1f}%<extra></extra>")
fig_a5.add_scatter(x=rank["mean"], y=rank["etq"], mode="markers",
                   name=f"ahora ({dekad_label(ULTIMA)})",
                   marker=dict(size=12, color="#b0413e"),
                   customdata=np.stack([rank["pct_gt40"], rank["km2"]], axis=-1),
                   hovertemplate="%{y}<br>ASI ahora %{x:.1f}%<br>"
                                 "%{customdata[0]:.0f}% del cultivo con ASI>40<br>"
                                 "%{customdata[1]:,.0f} km2 de cultivo<extra></extra>")
fig_a5.add_vline(x=40, line=dict(color="#ff8900", dash="dash"))
fig_a5.update_layout(height=680, margin=dict(l=240),
                     xaxis_title="ASI (% del area de cultivo del municipio)")
style_fig(fig_a5, "Los 20 municipios mas tensionados hoy",
           "Punto rojo: situacion en la ultima dekada publicada. Punto gris: su "
           "propio promedio de los 18 meses. Cuando el rojo esta muy a la derecha "
           "del gris, el deterioro es reciente.", y_source=-0.14)
fig_a5.show()


### 11.4 La dispersión entre municipios

Un promedio nacional bajo puede convivir con una cola de municipios en crisis. La caja muestra el rango intercuartil; la cola cuenta una historia distinta a la mediana.

In [ ]:
# --- A6 · como se mueve la distribucion, no solo el promedio -------------------
d6 = ASI18.dropna(subset=["mean"]).copy()
d6["etq"] = [dekad_label(c) for c in d6["dekad_id"]]
fig_a6 = px.box(d6, x="etq", y="mean", points=False, height=520,
                color_discrete_sequence=["#0b6fa4"],
                labels={"etq": "", "mean": "ASI municipal (%)"},
                category_orders={"etq": [dekad_label(c) for c in DK_18M
                                         if c in set(d6["dekad_id"])]})
fig_a6.add_hline(y=40, line=dict(color="#ff8900", width=1.2, dash="dot"))
fig_a6.update_xaxes(tickangle=-60, tickfont=dict(size=8))
style_fig(fig_a6, "Dispersion entre municipios en cada dekada",
           "Caja = rango intercuartil de los 290 municipios. Un promedio nacional "
           "bajo puede convivir con una cola de municipios en crisis: la mediana y "
           "la cola cuentan historias distintas.", y_source=-0.30, legend="off")
fig_a6.show()


### 11.5 Cuánto tiempo estuvo cada municipio bajo estrés

Porcentaje de las dekadas con dato en que el municipio superó el umbral. Identifica cronicidad, no episodios.

In [ ]:
# --- A7 · mapa acumulado del periodo ------------------------------------------
fig_a7 = continuous_map(persist, GEOJSON, "pct_dek_estres", "Cuanto tiempo estuvo cada municipio bajo estres", "% de las dekadas con dato (dentro de temporada) en que el ASI municipal "
    "supero 40 durante los ultimos 18 meses. Identifica cronicidad, no episodios.", scale=SCALE_ASI, value_range=(0, 100), bar_label="% de dekadas<br>con ASI>40", hover_extra={"asi_medio": ":.1f", "asi_max": ":.0f", "dek_con_dato": True})
fig_a7.show()


### 11.6 Los doce focos más crónicos

La misma información del mapa anterior en tabla, ordenada por proporción de dekadas bajo estrés.

In [ ]:
display(persist.nlargest(12, "pct_dek_estres")[
    ["adm2_name", "adm1_name", "asi_medio", "asi_max", "dek_estres",
     "dek_con_dato", "pct_dek_estres", "km2"]].round(1).reset_index(drop=True))


## 12. Contraste con la vegetación, perspectiva histórica y cierre

Cierra con el contraste entre ASI y VCI en la misma dekada, que evita los dos errores típicos: creer que no pasa nada donde el ASI no aplica, y confundir un exceso de agua con una sequía. Después sitúa el valor de hoy frente a la misma dekada de cada año desde 2005 y ordena los departamentos por estrés.

La tabla de resumen final y las limitaciones se construyen con f-strings sobre las mismas variables que alimentan las figuras, no con números escritos a mano: si un dato nuevo cambia una cifra, el texto cambia con ella.

*Produce:* tres figuras, la tabla de resumen, y la exportación a `./salidas` de diez figuras HTML y seis tablas en parquet y CSV.


### 12.1 El contraste entre estrés y vegetación

La misma dekada leída con los dos índices. El de vegetación cubre todo el territorio y todo el año, también fuera de temporada y fuera del área de cultivo, y por eso evita los dos errores típicos: creer que no pasa nada donde el otro índice no aplica, y confundir un exceso de agua con una sequía.

In [ ]:
# CELDA 12 · Contraste con vegetacion, lectura historica y cierre
# Explicacion, insumos y salidas: ver la seccion 12 del texto anterior.
# --- A8 · el ASI no basta: la misma dekada vista con VCI ----------------------
fig_a8 = class_map(ACTUAL_VCI, GEOJSON, "VCI", f"Condicion de la vegetacion · {dekad_label(ULTIMA)}", "El VCI cubre todo el territorio y todo el ano, tambien fuera de temporada y "
    "fuera del area de cultivo. Leer ASI y VCI juntos evita los dos errores "
    "tipicos: creer que no pasa nada donde el ASI no aplica, y confundir un "
    "exceso de agua con una sequia.", animation=None, hover_extra={"pct_lt0.35": ":.0f"})
fig_a8.show()


### 12.2 La dekada actual frente a la misma dekada de cada año

Cada barra es un año del periodo. La barra roja es el presente y las líneas punteadas son la mediana y el percentil 90 de su propia historia, lo que permite decir si el valor de hoy es normal o no.

In [ ]:
# --- A9 · lo de hoy, en perspectiva historica ---------------------------------
dek_actual = (int(ULTIMA[5:7]) - 1) * 3 + int(ULTIMA[-1])
serie_dek = asi_hist[asi_hist["dek_anio"] == dek_actual].dropna(subset=["valor"])
val_hoy = float(serie_dek[serie_dek["Year"] == int(ULTIMA[:4])]["valor"].iloc[0]) \
    if (serie_dek["Year"] == int(ULTIMA[:4])).any() \
    else float(nac18[nac18["dekad_id"] == ULTIMA]["mean"].iloc[0])
hist_prev = serie_dek[serie_dek["Year"] < int(ULTIMA[:4])]["valor"]
pctl = float((hist_prev < val_hoy).mean() * 100)

fig_a9 = go.Figure()
fig_a9.add_bar(x=serie_dek["Year"], y=serie_dek["valor"],
               marker_color=np.where(serie_dek["Year"] == int(ULTIMA[:4]),
                                     "#b0413e", "#c3cdd8"),
               hovertemplate="%{x}<br>ASI %{y:.1f}%<extra></extra>",
               name="ASI en esta dekada")
fig_a9.add_hline(y=float(hist_prev.median()), line=dict(color="#0b6fa4", dash="dot"),
                 annotation_text="mediana historica", annotation_position="top left")
fig_a9.add_hline(y=float(hist_prev.quantile(0.9)), line=dict(color="#e07b39", dash="dot"),
                 annotation_text="p90 historico", annotation_position="top left")
fig_a9.update_layout(height=460, yaxis_title="ASI nacional (%)", xaxis_title="")
style_fig(fig_a9, "La dekada actual comparada con la misma dekada de cada ano",
           f"{dekad_label(ULTIMA)} desde {START_YEAR} (n={len(hist_prev)} anios "
           f"previos). El valor de hoy ({val_hoy:.1f}%) esta "
           f"en el percentil {pctl:.0f} de su historia: "
           + ("por encima de lo normal." if pctl >= 66 else
              "dentro de lo normal." if pctl >= 33 else "por debajo de lo normal."),
           y_source=-0.22, legend="off")
fig_a9.show()


### 12.3 Departamentos ordenados por estrés

Media ponderada por área de cultivo de cada municipio del departamento. La línea vertical es el umbral de 40.

In [ ]:
# --- A10 · ranking departamental del momento ----------------------------------
dep_hoy = (dep18[dep18["dekad_id"] == ULTIMA].sort_values("mean")
           .merge(DEPT[["adm1_code", "adm1_name"]].drop_duplicates(),
                  on=["adm1_code", "adm1_name"], how="left"))
fig_a10 = px.bar(dep_hoy, x="mean", y="adm1_name", orientation="h", height=560,
                 color="mean", color_continuous_scale=SCALE_ASI,
                 range_color=(0, 100),
                 labels={"mean": "ASI (%)", "adm1_name": ""},
                 hover_data={"n_muni": True, "km2": ":,.0f"})
fig_a10.update_coloraxes(showscale=False)
fig_a10.add_vline(x=40, line=dict(color="#ff8900", dash="dash"))
style_fig(fig_a10, f"Departamentos ordenados por estres agricola · {dekad_label(ULTIMA)}",
           "Media ponderada por area de cultivo de cada municipio del departamento.",
           y_source=-0.16, legend="off")
fig_a10.show()


### 12.4 Tabla de resumen

Las cifras que resumen el cuaderno, construidas con f-strings sobre las mismas variables que alimentan las figuras. Si un dato nuevo cambia una cifra, esta tabla cambia con ella.

In [ ]:
# --- Resumen ------------------------------------------------------------------
asi_hoy = float(nac18[nac18["dekad_id"] == ULTIMA]["mean"].iloc[0])
vci_hoy = float(nac18_vci[nac18_vci["dekad_id"] == ULTIMA]["mean"].iloc[0])
peor_dek = nac18.loc[nac18["mean"].idxmax()]
resumen = pd.DataFrame([
    ("Ventana analizada", f"{dekad_label(DK_18M[0])} a {dekad_label(ULTIMA)} "
                          f"({len(DK_18M)} dekadas)"),
    ("Caso 1 · sequia", f"temporada primera 2019 · pico {dekad_label(PICO19)} · "
                        f"ASI nacional {nac19['mean'].max():.1f}% · "
                        f"{pico19['km2_gt40'].sum():,.0f} km2 de cultivo con ASI>40"),
    ("Caso 2 · exceso", f"Eta e Iota · {dekad_label(IMPACTO)} · lluvia "
                        f"{pico_lluvia['anom_pct']:+.0f}% sobre la LTA · VCI nacional "
                        f"{nac_vci.set_index('dekad_id').loc[ANTES, 'mean']:.2f} -> "
                        f"{nac_vci.set_index('dekad_id').loc[IMPACTO, 'mean']:.2f} (sin "
                        f"senal nacional) · {int((delta['caida'] > 0.15).sum())} "
                        "municipios con caida de VCI mayor a 0.15"),
    ("Estado actual (ASI)", f"{asi_hoy:.1f}% nacional · percentil {pctl:.0f} de su "
                            f"historia para esta dekada"),
    ("Estado actual (VCI)", f"{vci_hoy:.3f} nacional · "
                            f"{(ACTUAL_VCI['mean'] < 0.35).sum()} municipios bajo 0.35"),
    ("Peor dekada del periodo", f"{dekad_label(peor_dek['dekad_id'])} · ASI {peor_dek['mean']:.1f}%"),
    ("Focos cronicos", ", ".join(persist.nlargest(5, "pct_dek_estres")["adm2_name"])),
    ("Validacion", f"r={r:.3f} · MAE={mae:.2f} pp contra la serie oficial GIEWS"),
    ("Periodo historico", f"{START_YEAR} en adelante · linea base de la climatologia "
                          f"y de los percentiles"),
], columns=["", "resultado"])
display(resumen.style.hide(axis="index"))


### 12.5 Exportación y limitaciones

Escribe diez figuras HTML y seis tablas en parquet y CSV a `./salidas`, e imprime las limitaciones que conviene declarar al citar cualquiera de estos resultados fuera del cuaderno.

In [ ]:
# --- Exportacion --------------------------------------------------------------
SALIDA = Path("./salidas"); SALIDA.mkdir(exist_ok=True)
figuras = {"caso1_mapa_animado": fig_v1, "caso1_heatmap": fig_v2,
           "caso1_area_afectada": fig_v3, "caso2_lluvia": fig_w1,
           "caso2_mapa_animado": fig_w3, "caso2_dumbbell": fig_w4,
           "actual_mapa": fig_a1, "actual_tablero": fig_a3,
           "actual_persistencia": fig_a7, "actual_historico": fig_a9}
for nombre, f in figuras.items():
    f.write_html(SALIDA / f"{nombre}.html", include_plotlyjs="cdn")

tablas = {"panel_asi_2019": ASI19, "panel_vci_2020": VCI20,
          "panel_asi_18m": ASI18, "panel_vci_18m": vci_18m,
          "persistencia_18m": persist, "validacion": cmp_}
for nombre, t in tablas.items():
    t.to_parquet(SALIDA / f"{nombre}.parquet")
    t.to_csv(SALIDA / f"{nombre}.csv", index=False)

print(f"exportado a {SALIDA.resolve()} · {len(figuras)} figuras HTML y "
      f"{len(tablas)} tablas (parquet + csv)")
print("\nLimitaciones a tener presentes:")
print(" · el ASI solo existe dentro de la ventana de cultivo de cada temporada;")
print("   fuera de ella el municipio aparece sin dato, no en cero.")
print(f" · el analisis historico arranca en {START_YEAR}. La serie de lluvia de")
print("   GIEWS tiene un quiebre de homogeneidad alrededor de ese anio, asi que las")
print("   lecturas previas no son directamente comparables con las posteriores.")
print(f" · con la ventana desde {START_YEAR} la linea base de percentiles descansa en")
print(f"   {N_BASE} anios: suficiente para p10/p50/p90, escaso para colas extremas.")
print(" · GAUL 2015 no coincide en todos los limites con la division vigente;")
print("   los agregados municipales son comparables entre si, no oficiales.")


---

# Parte VI. Verificación

## 13. Chequeos numéricos de lo que afirman los textos

Toda afirmación de los subtítulos debería poder reproducirse en una línea. Esta celda las imprime todas juntas para revisión rápida: pico y superficie afectada del caso 1, anomalía de lluvia y caídas de VCI del caso 2, estado y focos crónicos de la ventana reciente, y las métricas de validación contra la serie oficial.

Si alguna cifra aquí no coincide con lo que dice una figura, la figura está mal, no el chequeo.


In [ ]:
# CELDA 13 · Chequeos numericos de lo que afirman los textos
# Explicacion, insumos y salidas: ver la seccion 13 del texto anterior.
print("CASO 1 - sequia primera 2019")
print(f"  pico nacional             : {dekad_label(PICO19)} · ASI {nac19['mean'].max():.1f}%")
print(f"  municipios con ASI>40     : {(pico19['mean'] > 40).sum()} · ASI>70: {(pico19['mean'] > 70).sum()}")
print(f"  km2 de cultivo con ASI>40 : {pico19['km2_gt40'].sum():,.0f} "
      f"({100 * pico19['km2_gt40'].sum() / pico19['km2'].sum():.1f}% del cultivo con dato)")
print(f"  ASI anual nacional 2019   : "
      f"{nac_anual[(nac_anual['anio'] == 2019)][['season', 'asi']].to_dict('records')}")

print("\nCASO 2 - Eta e Iota, noviembre 2020")
print(f"  maxima anomalia de lluvia : {dekad_label(pico_lluvia['dekad_id'])} "
      f"{pico_lluvia['anom_pct']:+.0f}% ({pico_lluvia['obs']:.0f} mm vs {pico_lluvia['lta']:.0f} mm)")
print(f"  VCI nacional antes/impacto/3 meses despues: "
      f"{[round(float(nac_vci.set_index('dekad_id').loc[k, 'mean']), 3) for k in (ANTES, IMPACTO, RECUP)]}")
print(f"  ASI postrera maximo en la ventana: {nac_asi20['mean'].max():.2f}% "
      "-> se mantiene en su rango habitual y responde al deficit del sur, no al "
      "exceso de agua del norte")
print(f"  municipios con caida de VCI >0.05: {int((delta['caida'] > 0.05).sum())} "
      f"· >0.15: {int((delta['caida'] > 0.15).sum())} de {len(delta)}")
print(f"  km2 de cultivo con VCI<0.35 en el impacto: "
      f"{VCI20[VCI20['dekad_id'] == IMPACTO]['km2_lt0.35'].sum():,.0f}")
print(f"  correlacion departamental lluvia-caida de VCI: r={rr:.2f} (nula)")

print("\nESTADO ACTUAL - ultimos 18 meses")
print(f"  ventana                   : {DK_18M[0]} a {ULTIMA} ({len(DK_18M)} dekadas)")
print(f"  ASI nacional hoy          : {asi_hoy:.1f}% (percentil {pctl:.0f} de esta dekada)")
print(f"  ASI nacional peor dekada  : {peor_dek['mean']:.1f}% en {dekad_label(peor_dek['dekad_id'])}")
print(f"  VCI nacional min/max      : {nac18_vci['mean'].min():.3f} / {nac18_vci['mean'].max():.3f}")
print(f"  municipios con ASI>40 hoy : {(ACTUAL['mean'] > 40).sum()} "
      f"· con VCI<0.35: {(ACTUAL_VCI['mean'] < 0.35).sum()}")
print(f"  focos cronicos (>25% de las dekadas con ASI>40): "
      f"{int((persist['pct_dek_estres'] > 25).sum())} municipios")

print("\nCALIDAD DEL DATO")
print(f"  periodo historico         : {START_YEAR} en adelante · linea base de "
      f"{N_BASE} anios")
print(f"  validacion vs GIEWS       : n={len(cmp_)} · r={r:.3f} · MAE={mae:.2f} pp · sesgo={sesgo:+.2f} pp")
print(f"  cobertura del pais        : {banderas.iloc[0, 0]:.1f}% de pixeles con dato valido de ASI")


---

## Limitaciones que conviene declarar al publicar

El ASI solo existe dentro de la ventana de cultivo de cada temporada. Fuera de ella el municipio aparece **sin dato**, no en cero, y tratarlo como cero fabrica una calma que el índice no afirma.

El análisis histórico arranca en 2005 por el quiebre de homogeneidad de la serie de lluvia. Eso deja la línea base de percentiles en catorce años: razonable para mediana y cuartiles, débil para caracterizar extremos. Ampliar `ANIO_INICIO` hacia atrás recupera muestra a costa de mezclar dos regímenes de medición.

Los límites de GAUL 2015 no coinciden en todos los casos con la división administrativa vigente. Los agregados municipales de este cuaderno son comparables entre sí, pero no son cifras oficiales de ninguna unidad territorial.

Los índices miden condición de la vegetación y déficit hídrico, no producción ni pérdida. El paso de estrés agroclimático a impacto en cosecha o en seguridad alimentaria requiere información que no está aquí.

## Cómo extender el cuaderno

Para mover el periodo histórico basta cambiar `ANIO_INICIO` en la sección 1. Para cambiar de país se ajustan `ISO3`, `PAIS_GAUL` y `HND_BBOX` en la misma sección, aunque habrá que revisar las temporadas agrícolas, que son específicas de Honduras. Para agregar un indicador se declara su rango válido en `VALID_RANGE` y se reutiliza `serie_municipal()` sin más cambios. Y si una cifra del análisis se va a citar fuera, conviene volver a correr la sección 4 primero: es la única que dice si el panel propio sigue reproduciendo el dato oficial de FAO.
